# Machine and operation classification

Run these cells from the repository root after installing `requirements.txt`.
The default split follows the anomaly notebook. Set `SPLIT_MODE` to
`"recording"` to evaluate on separate healthy recordings.

In [ ]:
from pathlib import Path
from time import perf_counter

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GroupShuffleSplit

from utils import data_loader_utils

WINDOW_SIZE = 4000
SAMPLE_RATE = 2000
RANDOM_STATE = 42
TRAIN_FRACTION = 0.80
SPLIT_MODE = "anomaly"
TARGETS = ("machine", "process")
AXES = ["X", "Y", "Z"]

data_path = Path("./data").resolve()
file_list = sorted(data_loader_utils.find_all_h5s_in_dir(data_path))

if not file_list:
    raise ValueError("No recordings found. Check the data directory.")

print("Recordings found:", len(file_list))

## Check one window

At 2 kHz, 4,000 samples span two seconds. X, Y, Z, and their combined magnitude
each contribute 13 time-domain and 12 frequency-domain features shared with the
anomaly detection branch, for 100 inputs. The Welch spectrum uses a Hann
window and 1,024-sample segments. Standard deviation uses the sample estimate.
Skew and excess kurtosis are set to zero for constant signals, where they are
undefined.

In [ ]:
file_df = data_loader_utils.format_file(file_list[0])
first_window = file_df.iloc[:WINDOW_SIZE][AXES].to_numpy()
if len(first_window) != WINDOW_SIZE:
    raise ValueError("The first recording has no complete two-second window.")

first_features = data_loader_utils.extract_window_features(first_window, fs=SAMPLE_RATE)
print("Window shape:", first_window.shape)
pd.DataFrame([first_features])

## Build the feature table

Use every complete, non-overlapping two-second window from each recording.
Drop any shorter tail. Time spent reading and formatting HDF5 files is
reported separately from overall feature-table preparation.

In [ ]:
preparation_start = perf_counter()
read_seconds = 0.0
rows = []
skipped_tails = 0

for file_path in file_list:
    path = Path(file_path)
    read_start = perf_counter()
    recording = data_loader_utils.format_file(path)
    read_seconds += perf_counter() - read_start

    vibration = recording[AXES].to_numpy()
    window_count, tail_samples = divmod(len(vibration), WINDOW_SIZE)
    skipped_tails += bool(tail_samples)
    if not window_count:
        continue

    metadata = {
        "source_file": path.relative_to(data_path).as_posix(),
        "filename": path.name,
        "healthy": path.parent.name == "good",
        "machine": int(recording["machine"].iloc[0]),
        "process": int(recording["process"].iloc[0]),
    }
    for window_id in range(window_count):
        start = window_id * WINDOW_SIZE
        row = data_loader_utils.extract_window_features(
            vibration[start:start + WINDOW_SIZE], fs=SAMPLE_RATE
        )
        rows.append({**row, **metadata, "window": window_id})

if not rows:
    raise ValueError("No complete windows found. Check the data directory.")

features_df = pd.DataFrame(rows)
print("Complete windows:", len(features_df))
print("Recordings with complete windows:", features_df["source_file"].nunique())
print("Incomplete tails skipped:", skipped_tails)
print(f"HDF5 reading and formatting: {read_seconds:.1f} s")
print(f"Feature-table preparation including HDF5 reads: {perf_counter() - preparation_start:.1f} s")
features_df.head()

## Split the data

For the anomaly split, sample 80% of all windows from healthy windows only,
using seed 42. Divide the remaining healthy windows and the filename-sorted
faulty windows into selection and final test sets. The feature table uses
sorted input paths so reruns reproduce the split. The anomaly notebook uses
filesystem order, which can change the windows drawn with the same seed.

Windows from the same recording can cross these sets. The recording split
holds out 20% of healthy recordings to measure performance on unseen files.
Machine, operation, file, health, and window metadata stay out of the inputs.

In [ ]:
feature_columns = list(first_features)
healthy = features_df.loc[features_df["healthy"]]

if SPLIT_MODE == "anomaly":
    faulty = features_df.loc[~features_df["healthy"]].sort_values("filename", kind="stable")
    train_count = round(TRAIN_FRACTION * len(features_df))
    if not 0 < train_count < len(healthy):
        raise ValueError("The anomaly split needs enough healthy windows for training and holdout.")
    train = healthy.sample(n=train_count, random_state=RANDOM_STATE)
    healthy_holdout = healthy.drop(train.index)
    healthy_midpoint = len(healthy_holdout) // 2
    faulty_midpoint = len(faulty) // 2
    evaluation_sets = {
        "selection": pd.concat([healthy_holdout.iloc[:healthy_midpoint], faulty.iloc[:faulty_midpoint]]),
        "final test": pd.concat([healthy_holdout.iloc[healthy_midpoint:], faulty.iloc[faulty_midpoint:]]),
    }
elif SPLIT_MODE == "recording":
    splitter = GroupShuffleSplit(
        n_splits=1, test_size=1 - TRAIN_FRACTION, random_state=RANDOM_STATE
    )
    train_idx, test_idx = next(splitter.split(healthy, groups=healthy["source_file"]))
    train = healthy.iloc[train_idx]
    test = healthy.iloc[test_idx]
    assert set(train["source_file"]).isdisjoint(test["source_file"])
    evaluation_sets = {"test": test}
else:
    raise ValueError(f"Unknown split mode: {SPLIT_MODE}")

training_files = set(train["source_file"])
for split_name, partition in {"train": train, **evaluation_sets}.items():
    health_counts = partition["healthy"].value_counts()
    print(
        f"{split_name}: {len(partition)} windows "
        f"({health_counts.get(True, 0)} healthy, {health_counts.get(False, 0)} faulty) "
        f"from {partition['source_file'].nunique()} recordings"
    )
    if split_name != "train":
        shared_files = training_files.intersection(partition["source_file"])
        print(f"  Recordings also in training: {len(shared_files)}")

for target in TARGETS:
    missing_classes = set(features_df[target]) - set(train[target])
    if missing_classes:
        raise ValueError(f"{target}: classes absent from training: {sorted(missing_classes)}")

## Train and evaluate the classifiers

Start with the forest settings from the highest-PR-AUC Isolation Forest trial
in [the anomaly notebook](https://github.com/Mageto-Nyakoni/cnc-machine-fault-detection/blob/47955474ec2e8a6a7ad1fd757859f8851fd49fcc/anomaly_detection.ipynb).
Each Random Forest has 300 trees and draws up to 1,024 training windows per tree
with replacement. It considers 70% of the features at each split.

Both classifiers use the same training windows. Fit times exclude prediction.
Train on healthy windows and report machine and operation predictions for
each holdout. The anomaly split includes faulty windows in those reports.

In [ ]:
models = {}
X_train = train[feature_columns]

for target in TARGETS:
    model = RandomForestClassifier(
        n_estimators=300,
        bootstrap=True,
        max_samples=min(1024, len(train)),
        max_features=0.7,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    fit_start = perf_counter()
    model.fit(X_train, train[target])
    print(f"\n{target} fit time: {perf_counter() - fit_start:.1f} s")
    models[target] = model

    for split_name, partition in evaluation_sets.items():
        actual = partition[target]
        predictions = model.predict(partition[feature_columns])
        print(f"\n{target} classification on {split_name}")
        print(f"Accuracy: {accuracy_score(actual, predictions):.3f}")
        missing = set(model.classes_) - set(actual)
        if missing:
            print(f"Classes without test windows: {sorted(missing)}")
        print(classification_report(
            actual,
            predictions,
            labels=sorted(actual.unique()),
            digits=3,
            zero_division=0,
        ))